# Importing Dependencies

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Loading Dataset

In [2]:
df = pd.read_csv('train.txt',sep = ';',header = None,names = ['text','emotion'])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

,0
text,0
emotion,0


# Encoding unique Emotions

In [5]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
  emotion_numbers[emo] = i
  i +=1

df['emotion'] = df['emotion'].map(emotion_numbers)

In [6]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [7]:
# LowerCase all the sentences.
df['text'] = df['text'].apply(lambda x : x.lower())

In [8]:
# Removing Punctuations
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))


In [9]:
df['text'] = df['text'].apply(remove_punc)

In [10]:
# Removing numbers
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [11]:
# Removing Emojis
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [12]:
import nltk

In [13]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [14]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [15]:
# Removing Stop words.
stop_words = set(stopwords.words('english'))
# word_tokens = df['text'].apply(lambda x : word_tokenize(x))

In [16]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [18]:
def tokenize_text(text):
  return word_tokenize(text)

# Fix: Download the missing 'punkt_tab' resource
import nltk
nltk.download('punkt_tab')

df['text'] = df['text'].apply(tokenize_text)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [21]:
df.head()

,text,emotion
0,"[i, didnt, feel, humiliated]",0
1,"[i, can, go, from, feeling, so, hopeless, to, ...",0
2,"[im, grabbing, a, minute, to, post, i, feel, g...",1
3,"[i, am, ever, feeling, nostalgic, about, the, ...",2
4,"[i, am, feeling, grouchy]",1


In [22]:
df.loc[1]['text']

['i',
 'can',
 'go',
 'from',
 'feeling',
 'so',
 'hopeless',
 'to',
 'so',
 'damned',
 'hopeful',
 'just',
 'from',
 'being',
 'around',
 'someone',
 'who',
 'cares',
 'and',
 'is',
 'awake']

In [23]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)
  return ' '.join(cleaned)

# Convert list of words back to string before applying the remove function
df['text'] = df['text'].apply(lambda x: ' '.join(x))
df['text'] = df['text'].apply(remove)

In [24]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [25]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [27]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.7678125


In [28]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

In [29]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [30]:
y_pred = nb2_model.predict(X_test_tfidf)

In [31]:
print(accuracy_score(y_test, y_pred))

0.6609375


In [32]:
from sklearn.linear_model import LogisticRegression

In [33]:
logistic_model = LogisticRegression(max_iter=1000)

In [34]:
logistic_model.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [35]:
log_pred = logistic_model.predict(X_test_tfidf)

In [36]:
print(accuracy_score(y_test,log_pred ))

0.8615625


In [63]:
print(emotion_numbers)

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}


In [60]:
# Making a predictive model
input_text =["i feel romantic too"]
# Converting the text into tokens
input_data_feature = tfidf_vectorizer.transform(input_text)
# Making prediction
prediction = logistic_model.predict(input_data_feature)
print(prediction)

if (prediction[0]==0):
  print('sadness')
elif (prediction[0]==1):
  print('anger')
elif (prediction[0]==2):
  print('love')
elif (prediction[0]==3):
  print('surprise')
elif (prediction[0]==4):
  print('fear')
else:
  print('joy')

[2]
love


In [69]:
# Making a predictive model
# Get input text from the user
raw_input_text = input("Enter a sentence to predict its emotion: ")

# Preprocess the input text using existing functions
processed_text = raw_input_text.lower()
processed_text = remove_punc(processed_text)
processed_text = remove_numbers(processed_text)
processed_text = remove_emojis(processed_text)
# The 'remove' function handles stop word removal and joining into a single string
processed_text = remove(processed_text)

# Convert preprocessed text into vectors
# tfidf_vectorizer.transform expects a list of strings
input_data_feature = tfidf_vectorizer.transform([processed_text])

# Predict
prediction = logistic_model.predict(input_data_feature)
predicted_label = prediction[0]

# Get the emotion name from the dictionary
predicted_emotion_name = emotion_names[predicted_label]

print(f"The predicted emotion for the sentence is: {predicted_emotion_name}")

Enter a sentence to predict its emotion: I am terrified of heights
The predicted emotion for the sentence is: fear


In [73]:
# Making a predictive model
# Get input text from the user
raw_input_text = input("Enter a sentence to predict its emotion: ")

# Preprocess the input text using existing functions
processed_text = raw_input_text.lower()
processed_text = remove_punc(processed_text)
processed_text = remove_numbers(processed_text)
processed_text = remove_emojis(processed_text)
# The 'remove' function handles stop word removal and joining into a single string
processed_text = remove(processed_text)

# Convert preprocessed text into vectors
# tfidf_vectorizer.transform expects a list of strings
input_data_feature = tfidf_vectorizer.transform([processed_text])

# Predict
prediction = logistic_model.predict(input_data_feature)
predicted_label = prediction[0]

# Get the emotion name from the dictionary
predicted_emotion_name = emotion_names[predicted_label]

print(f"The predicted emotion for the sentence is: {predicted_emotion_name}")

Enter a sentence to predict its emotion: I'm pumped about this game!
The predicted emotion for the sentence is: joy


In [72]:
# Making a predictive model
# Get input text from the user
raw_input_text = input("Enter a sentence to predict its emotion: ")

# Preprocess the input text using existing functions
processed_text = raw_input_text.lower()
processed_text = remove_punc(processed_text)
processed_text = remove_numbers(processed_text)
processed_text = remove_emojis(processed_text)
# The 'remove' function handles stop word removal and joining into a single string
processed_text = remove(processed_text)

# Convert preprocessed text into vectors
# tfidf_vectorizer.transform expects a list of strings
input_data_feature = tfidf_vectorizer.transform([processed_text])

# Predict
prediction = logistic_model.predict(input_data_feature)
predicted_label = prediction[0]

# Get the emotion name from the dictionary
predicted_emotion_name = emotion_names[predicted_label]

print(f"The predicted emotion for the sentence is: {predicted_emotion_name}")

Enter a sentence to predict its emotion: It makes me so mad when people lie to me
The predicted emotion for the sentence is: anger


In [76]:
# Making a predictive model
# Get input text from the user
raw_input_text = input("Enter a sentence to predict its emotion: ")

# Preprocess the input text using existing functions
processed_text = raw_input_text.lower()
processed_text = remove_punc(processed_text)
processed_text = remove_numbers(processed_text)
processed_text = remove_emojis(processed_text)
# The 'remove' function handles stop word removal and joining into a single string
processed_text = remove(processed_text)

# Convert preprocessed text into vectors
# tfidf_vectorizer.transform expects a list of strings
input_data_feature = tfidf_vectorizer.transform([processed_text])

# Predict
prediction = logistic_model.predict(input_data_feature)
predicted_label = prediction[0]

# Get the emotion name from the dictionary
predicted_emotion_name = emotion_names[predicted_label]

print(f"The predicted emotion for the sentence is: {predicted_emotion_name}")

Enter a sentence to predict its emotion: i too feel as if i am a stranger in a strange land and i am raising my son in a place that is not his father s ancestral home
The predicted emotion for the sentence is: surprise
